# Getting started with Optimization


This notebook starts with a tiny local DTLZ2 evaluator, then shows the common AID2E optimizer interface, and finally branches into Ax and PyMOO-specific workflows. The last section moves away from YAML and defines optimizer settings inline so the same ideas can be reused interactively.


In [12]:
from __future__ import annotations

import math
from pathlib import Path
from pprint import pprint

from aid2e.utilities import build_optimizer_from_config
from aid2e.utilities.configurations import load_config
from aid2e.utilities.configurations.optimizer_config import OptimizerConfiguration

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "dtlz2_ax_optimizer_only.yml").exists():
    candidate = NOTEBOOK_DIR / "examples" / "optimizers"
    if candidate.exists():
        NOTEBOOK_DIR = candidate.resolve()
    else:
        raise FileNotFoundError(
            "Run this notebook from examples/dtlz2/optimizers or from the repository root."
        )

AX_CONFIG_PATH = NOTEBOOK_DIR / "dtlz2_ax_optimizer_only.yml"
PYMOO_CONFIG_PATH = NOTEBOOK_DIR / "dtlz2_pymoo_optimizer_only.yml"

ax_full_config = load_config(str(AX_CONFIG_PATH))
pymoo_full_config = load_config(str(PYMOO_CONFIG_PATH))

print("Notebook directory:", NOTEBOOK_DIR)
print("Ax config:", AX_CONFIG_PATH.name)
print("PyMOO config:", PYMOO_CONFIG_PATH.name)


Notebook directory: /sciclone/scr10/ksuresh/AID2E-framework/examples/dtlz2/optimizers
Ax config: dtlz2_ax_optimizer_only.yml
PyMOO config: dtlz2_pymoo_optimizer_only.yml


We keep the evaluator local on purpose. That way the notebook focuses on the optimizer contract instead of workflow orchestration.

In this notebook, the local evaluator uses the **DTLZ2** benchmark problem, a standard multi-objective test function designed to check whether an optimizer can recover a smooth **Pareto front**.

For a problem with $m$ objectives and $n$ decision variables, let the decision vector be

$$
\mathbf{x} = (x_1, x_2, \dots, x_n), \qquad 0 \le x_i \le 1.
$$

DTLZ2 defines

$$
g(\mathbf{x}_m) = \sum_{i=m}^{n} (x_i - 0.5)^2,
$$

where $\mathbf{x}_m$ denotes the tail of the vector used in the distance term. The objectives are then

$$
f_j(\mathbf{x}) = (1 + g(\mathbf{x}_m))
\left(
\prod_{i=1}^{m-j} \cos\left(\frac{\pi}{2} x_i\right)
\right)
$$

for $j = 1, \dots, m$$, with the final sine factor applied in the standard DTLZ2 construction:

$$
f_j(\mathbf{x}) =
(1 + g(\mathbf{x}_m))
\left(
\prod_{i=1}^{m-j} \cos\left(\frac{\pi}{2} x_i\right)
\right)
\sin\left(\frac{\pi}{2} x_{m-j+1}\right)
\quad \text{for } j > 1.
$$

For the **two-objective** case used here, this simplifies to

$$
g = \sum_{i=2}^{n} (x_i - 0.5)^2,
$$

$$
f_1 = (1 + g)\cos\left(\frac{\pi}{2}x_1\right),
\qquad
f_2 = (1 + g)\sin\left(\frac{\pi}{2}x_1\right).
$$

In this notebook, $$n = 5$$, so the evaluator uses

$$
g = (x_2 - 0.5)^2 + (x_3 - 0.5)^2 + (x_4 - 0.5)^2 + (x_5 - 0.5)^2.
$$

The optimization goal is to **minimize both** $$f_1$$ and $$f_2$$. When $$g = 0$$, the solutions lie on the Pareto-optimal front, which for the two-objective case forms a quarter of the unit circle:

$$
f_1^2 + f_2^2 = 1, \qquad f_1 \ge 0,\; f_2 \ge 0.
$$

That makes DTLZ2 a useful example because it has a simple analytical form while still testing multi-objective behavior clearly.

In [36]:
def dtlz2_objectives(parameters: dict[str, float]) -> dict[str, float]:
    """ 
    Compute the two-objective DTLZ2 function from a parameter dict.
    Remember that the parameters dictonary will always have keys in the format DTLZ2_variables.x{N}, Refer the design.params.
    """
    x1 = float(parameters["DTLZ2_variables.x1"])
    tail = [
        float(parameters["DTLZ2_variables.x2"]),
        float(parameters["DTLZ2_variables.x3"]),
        float(parameters["DTLZ2_variables.x4"]),
        float(parameters["DTLZ2_variables.x5"]),
    ]
    g = sum((value - 0.5) ** 2 for value in tail)
    factor = 1.0 + g
    f1 = factor * math.cos(x1 * math.pi / 2.0)
    f2 = factor * math.sin(x1 * math.pi / 2.0)
    return {"f1": float(f1), "f2": float(f2)}


def evaluate_candidates(optimizer, candidates: list[dict[str, float]], phase: str):
    """Evaluate suggested candidates locally and push results back into the optimizer."""
    start_index = len(optimizer.get_trials()) - len(candidates)
    records = []
    for offset, parameters in enumerate(candidates):
        trial_index = start_index + offset
        metrics = dtlz2_objectives(parameters)
        optimizer.update_with_results(
            trial_index=trial_index,
            parameters=parameters,
            metrics=metrics,
        )
        _record = {
            "trial_index": trial_index,
            "phase": phase,
        } | parameters | metrics
        records.append(_record)
    return records


def summarize_optimizer(optimizer, label: str) -> dict[str, int | str]:
    results = optimizer.get_optimization_results()
    pareto_front = optimizer.get_pareto_front()
    return {
        "label": label,
        "n_trials": results["n_trials"],
        "pareto_points": len(pareto_front),
    }


Lets quickly check by providing a dummy input. To make sure everything works

In [37]:

design_cfg = ax_full_config.problem.design_config

demo_parameters = {
    name: 0.5
    for name in design_cfg.get_parameter_names()
}

demo_parameters.update(
    dict(
        zip(
            sorted(demo_parameters),
            [0.25, 0.50, 0.75, 0.50, 0.50],
            strict=False,
        )
    )
)

print("DTLZ2 objectives:")
pprint(dtlz2_objectives(demo_parameters))


DTLZ2 objectives:
{'f1': 0.9816220032932421, 'f2': 0.4066011468879079}


The common optimizer contract is the same across backends: load config, build the optimizer, call `suggest_candidates(...)`, evaluate locally, then return metrics through `update_with_results(...)`.


In [38]:
generic_problem = ax_full_config.problem
generic_optimizer_cfg = ax_full_config.optimizer

print("Problem:", generic_problem.name)
print("Objectives:", [objective.name for objective in generic_problem.objectives])
print("Design parameters:", generic_problem.design_config.get_parameter_names())
print("Optimizer section:")
pprint(generic_optimizer_cfg.model_dump())


Problem: DTLZ2 Optimizer-Only Ax Example
Objectives: ['f1', 'f2']
Design parameters: ['DTLZ2_variables.x1', 'DTLZ2_variables.x2', 'DTLZ2_variables.x3', 'DTLZ2_variables.x4', 'DTLZ2_variables.x5']
Optimizer section:
{'name': 'ax',
 'parameters': {'batch_size': 2,
                'generator': 'BOTORCH_MODULAR',
                'generator_gen_kwargs': {'model_gen_options': {'optimizer_kwargs': {'num_restarts': 10,
                                                                                    'sequential': False}}},
                'generator_kwargs': {'acquisition_options': {'prune_baseline': True},
                                     'botorch_acqf_class': 'qLogNoisyExpectedHypervolumeImprovement'},
                'initialization_strategy': 'sobol',
                'n_initial_samples': 4,
                'n_iterations': 3,
                'objective_thresholds': {'f1': 1.0, 'f2': 1.0},
                'seed': 42},
 'type': 'bayesian'}


In [39]:
generic_optimizer = build_optimizer_from_config(generic_problem, generic_optimizer_cfg)
generic_records = []

for step in range(2):
    candidates = generic_optimizer.suggest_candidates(n_candidates=1)
    generic_records.extend(
        evaluate_candidates(generic_optimizer, candidates, phase=f"generic-step-{step + 1}")
    )

print("Recorded trials from the generic loop:")
pprint(generic_records)
print("Generic loop summary:")
pprint(summarize_optimizer(generic_optimizer, "generic-ax-demo"))


Recorded trials from the generic loop:
[{'DTLZ2_variables.x1': 0.9975133538246155,
  'DTLZ2_variables.x2': 0.10436639189720154,
  'DTLZ2_variables.x3': 0.8229788541793823,
  'DTLZ2_variables.x4': 0.4194321632385254,
  'DTLZ2_variables.x5': 0.5283221006393433,
  'f1': 0.004953339804529975,
  'f2': 1.268124935891223,
  'phase': 'generic-step-1',
  'trial_index': 0},
 {'DTLZ2_variables.x1': 0.39916136860847473,
  'DTLZ2_variables.x2': 0.5109770465642214,
  'DTLZ2_variables.x3': 0.3220283752307296,
  'DTLZ2_variables.x4': 0.5257955053821206,
  'DTLZ2_variables.x5': 0.4753276174888015,
  'f1': 0.8365691769151665,
  'f2': 0.6061209438136624,
  'phase': 'generic-step-2',
  'trial_index': 1}]
Generic loop summary:
{'label': 'generic-ax-demo', 'n_trials': 2, 'pareto_points': 2}


Now we follow the full Ax optimizer-only example. The loop uses an explicit initialization phase and then a model-based phase.


In [40]:
def run_ax_optimization(full_config):
    optimizer_config = full_config.optimizer.parse_algorithm_params()
    optimizer = build_optimizer_from_config(full_config.problem, full_config.optimizer)
    records = []

    remaining_init = optimizer_config.n_initial_samples
    while remaining_init > 0:
        current_batch = min(optimizer_config.batch_size, remaining_init)
        candidates = optimizer.suggest_candidates(n_candidates=current_batch)
        records.extend(evaluate_candidates(optimizer, candidates, phase="init"))
        remaining_init -= current_batch

    for iteration in range(optimizer_config.n_iterations):
        candidates = optimizer.suggest_candidates(
            n_candidates=optimizer_config.batch_size
        )
        records.extend(
            evaluate_candidates(optimizer, candidates, phase=f"iter-{iteration + 1}")
        )

    return optimizer_config, optimizer, records


ax_optimizer_config, ax_optimizer, ax_records = run_ax_optimization(ax_full_config)
print("First three Ax records:")
pprint(ax_records[:3])
print("Total Ax records:", len(ax_records))


/sciclone/home/ksuresh/.conda/envs/env_AID2E/lib/python3.11/site-packages/gpytorch/likelihoods/noise_models.py:150: NumericalWarning: Very small noise values detected. This will likely lead to numerical instabilities. Rounding small noise values up to 1e-06.
  warnings.warn(
/sciclone/home/ksuresh/.conda/envs/env_AID2E/lib/python3.11/site-packages/gpytorch/likelihoods/noise_models.py:150: NumericalWarning: Very small noise values detected. This will likely lead to numerical instabilities. Rounding small noise values up to 1e-06.
  warnings.warn(
/sciclone/home/ksuresh/.conda/envs/env_AID2E/lib/python3.11/site-packages/gpytorch/likelihoods/noise_models.py:150: NumericalWarning: Very small noise values detected. This will likely lead to numerical instabilities. Rounding small noise values up to 1e-06.
  warnings.warn(


First three Ax records:
[{'DTLZ2_variables.x1': 0.9975133538246155,
  'DTLZ2_variables.x2': 0.10436639189720154,
  'DTLZ2_variables.x3': 0.8229788541793823,
  'DTLZ2_variables.x4': 0.4194321632385254,
  'DTLZ2_variables.x5': 0.5283221006393433,
  'f1': 0.004953339804529975,
  'f2': 1.268124935891223,
  'phase': 'init',
  'trial_index': 0},
 {'DTLZ2_variables.x1': 0.39916136860847473,
  'DTLZ2_variables.x2': 0.5109770465642214,
  'DTLZ2_variables.x3': 0.3220283752307296,
  'DTLZ2_variables.x4': 0.5257955053821206,
  'DTLZ2_variables.x5': 0.4753276174888015,
  'f1': 0.8365691769151665,
  'f2': 0.6061209438136624,
  'phase': 'init',
  'trial_index': 1},
 {'DTLZ2_variables.x1': 0.18095120228827,
  'DTLZ2_variables.x2': 0.36145188845694065,
  'DTLZ2_variables.x3': 0.5728367641568184,
  'DTLZ2_variables.x4': 0.14431990031152964,
  'DTLZ2_variables.x5': 0.22098449897021055,
  'f1': 1.1795517337847434,
  'f2': 0.3446034690561871,
  'phase': 'init',
  'trial_index': 2}]
Total Ax records: 10


In [41]:
ax_summary = summarize_optimizer(ax_optimizer, "ax")
ax_summary["generator"] = ax_optimizer_config.generator
pprint(ax_summary)


{'generator': 'BOTORCH_MODULAR',
 'label': 'ax',
 'n_trials': 10,
 'pareto_points': 4}


In [42]:
print("Generation strategy name:", ax_optimizer.generation_strategy.name)
nodes = getattr(ax_optimizer.generation_strategy, "nodes", None)
if nodes:
    print("Generation strategy nodes:", [node.name for node in nodes])
else:
    print(ax_optimizer.generation_strategy)


Generation strategy name: Sobol+ModularBoTorch
GenerationStrategy(name='Sobol+ModularBoTorch', nodes=[GenerationNode(name='Sobol', generator_specs=[GeneratorSpec(generator_enum=Sobol, generator_key_override=None)], transition_criteria=[MinTrials(transition_to='ModularBoTorch')], pausing_criteria=None), GenerationNode(name='ModularBoTorch', generator_specs=[GeneratorSpec(generator_enum=BoTorch, generator_key_override=None)], transition_criteria=None, pausing_criteria=None)])


PyMOO uses the same outer interface, but the inner optimization model is generation-based. In this configuration the algorithm is inferred automatically.


In [43]:
def run_pymoo_optimization(full_config):
    optimizer_config = full_config.optimizer.parse_algorithm_params()
    optimizer = build_optimizer_from_config(full_config.problem, full_config.optimizer)
    records = []

    for generation in range(optimizer_config.n_iterations):
        candidates = optimizer.suggest_candidates()
        records.extend(
            evaluate_candidates(optimizer, candidates, phase=f"gen-{generation + 1}")
        )

    return optimizer_config, optimizer, records


pymoo_optimizer_config, pymoo_optimizer, pymoo_records = run_pymoo_optimization(
    pymoo_full_config
)
print("First three PyMOO records:")
pprint(pymoo_records[:3])
print("Total PyMOO records:", len(pymoo_records))


First three PyMOO records:
[{'DTLZ2_variables.x1': 0.7739560485559633,
  'DTLZ2_variables.x2': 0.4388784397520523,
  'DTLZ2_variables.x3': 0.8585979199113825,
  'DTLZ2_variables.x4': 0.6973680290593639,
  'DTLZ2_variables.x5': 0.09417734788764953,
  'f1': 0.46445830050774667,
  'f2': 1.2526397290110498,
  'phase': 'gen-1',
  'trial_index': 0},
 {'DTLZ2_variables.x1': 0.9756223516367559,
  'DTLZ2_variables.x2': 0.761139701990353,
  'DTLZ2_variables.x3': 0.7860643052769538,
  'DTLZ2_variables.x4': 0.12811363267554587,
  'DTLZ2_variables.x5': 0.45038593789556713,
  'f1': 0.04941518013165959,
  'f2': 1.2898415294878025,
  'phase': 'gen-1',
  'trial_index': 1},
 {'DTLZ2_variables.x1': 0.37079802423258124,
  'DTLZ2_variables.x2': 0.9267649888486018,
  'DTLZ2_variables.x3': 0.6438651200806645,
  'DTLZ2_variables.x4': 0.82276161327083,
  'DTLZ2_variables.x5': 0.44341419882733113,
  'f1': 1.0941743623514157,
  'f2': 0.7207032409991352,
  'phase': 'gen-1',
  'trial_index': 2}]
Total PyMOO record

In [44]:
pymoo_summary = summarize_optimizer(pymoo_optimizer, "pymoo")
pymoo_summary["resolved_algorithm"] = pymoo_optimizer.resolved_algorithm
pprint(pymoo_summary)


{'label': 'pymoo',
 'n_trials': 32,
 'pareto_points': 9,
 'resolved_algorithm': 'nsga2'}


Once both runs are complete, it is easy to compare them at a high level before digging into backend-specific details.


In [45]:
comparison = {
    "ax": ax_summary,
    "pymoo": pymoo_summary,
}
pprint(comparison)


{'ax': {'generator': 'BOTORCH_MODULAR',
        'label': 'ax',
        'n_trials': 10,
        'pareto_points': 4},
 'pymoo': {'label': 'pymoo',
           'n_trials': 32,
           'pareto_points': 9,
           'resolved_algorithm': 'nsga2'}}


For more interactive work, you can keep the problem definition from YAML and define only the optimizer settings inline. Here we make that inline section more opinionated by choosing a SAASBO-style Ax surrogate with qNEHVI, and an explicit NSGA2 setup for PyMOO.


In [ ]:
ax_inline_optimizer_payload = {
        "name": "ax",
        "type": "bayesian",
        "parameters": {
            "initialization_strategy": "sobol",
            "generator": "BOTORCH_MODULAR",
            "generator_kwargs": {
                "surrogate_spec": {
                    "model_configs": [
                        {
                            "botorch_model_class": "SaasFullyBayesianSingleTaskGP"
                        }
                    ]
                },
                "botorch_acqf_class": "qLogNoisyExpectedHypervolumeImprovement",
            },
            "objective_thresholds": {"f1": 1.0, "f2": 1.0},
            "n_initial_samples": 10,
            "n_iterations": 20,
            "batch_size": 4,
            "seed": 7,
        },
    }
ax_inline_optimizer_cfg = OptimizerConfiguration(**ax_inline_optimizer_payload)
ax_inline_optimizer = build_optimizer_from_config(
    ax_full_config.problem,
    ax_inline_optimizer_cfg,
)



Inline Ax optimizer payload (SAAS surrogate + qNEHVI):
{'name': 'ax',
 'parameters': {'batch_size': 1,
                'generator': 'BOTORCH_MODULAR',
                'generator_kwargs': {'botorch_acqf_class': 'qNoisyExpectedHypervolumeImprovement',
                                     'surrogate_spec': {'model_configs': [{'botorch_model_class': 'SaasFullyBayesianSingleTaskGP'}]}},
                'initialization_strategy': 'sobol',
                'n_initial_samples': 2,
                'n_iterations': 1,
                'objective_thresholds': {'f1': 1.0, 'f2': 1.0},
                'seed': 7},
 'type': 'bayesian'}
First Ax candidate from the inline config:
{'DTLZ2_variables.x1': 0.19947312772274017,
 'DTLZ2_variables.x2': 0.17093220353126526,
 'DTLZ2_variables.x3': 0.7493569254875183,
 'DTLZ2_variables.x4': 0.18431377410888672,
 'DTLZ2_variables.x5': 0.5505224466323853}


In [ ]:
from pprint import pformat
from typing import Any

records: list[dict[str, Any]] = []
first_candidate: dict[str, Any] | None = None
while remaining_init > 0:
    init_round += 1
    current_batch = min(ax_inline_optimizer_cfg.batch_size, remaining_init)
    candidates = ax_inline_optimizer.suggest_candidates(n_candidates=current_batch)
    if first_candidate is None and candidates:
        first_candidate = dict(candidates[0])
        print("First Ax candidate from the inline config:")
        print(pformat(first_candidate))
    records.extend(
        evaluate_candidates(
            ax_inline_optimizer,
            candidates,
            phase=f"init-{init_round}",
        )
    )
    remaining_init -= current_batch

for iteration in range(ax_inline_optimizer_cfg.n_iterations):
    candidates = ax_inline_optimizer.suggest_candidates(
        n_candidates=ax_inline_optimizer_cfg.batch_size
    )
    records.extend(
        evaluate_candidates(
            ax_inline_optimizer,
            candidates,
            phase=f"iter-{iteration + 1}",
        )
    )